# 01: Fokker-Planck PDE Solver Introduction

This notebook demonstrates the core Fokker-Planck PDE solver with setup, solving, diagnostics, and visualization.

In [4]:
# Setup and imports
import sys, os

repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if os.path.isdir(src_path) and src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import matplotlib.pyplot as plt
from stochlib import configure_logging, get_logger
import logging

# Configure logging
configure_logging(level=logging.INFO, verbose=True)
logger = get_logger("notebook")
logger.info("Starting FP introduction notebook")

2026-01-19 23:45:03,258 [INFO    ] stochlib.notebook: Starting FP introduction notebook


In [5]:
# Import FP components
from stochlib.setup import Grid, VelocitiesConfig, DiffusionConfig, InitialCondition
from stochlib.boundary_conditions import BoundaryConditions
from stochlib.fokker_planck import FokkerPlanckSolver, SolutionDiagnostics
from stochlib.fokker_planck.plotting import plot_1d_distribution, plot_2d_distribution

logger.info("Imported FP components")

2026-01-19 23:45:12,761 [INFO    ] stochlib.notebook: Imported FP components


## Part 1: 1D Diffusion Example

In [7]:
# Create 1D grid
grid_1d = Grid(x_start=-10.0, x_end=10.0, num_points_x=301)
logger.info(f"Created 1D grid: {grid_1d.num_points_x} points")

# Initial condition: Gaussian with mean 0, std 1.0
ic = InitialCondition(grid_1d, func_type="gaussian", x0=0.0, sigma_x=1.0)
logger.info(f"Initial condition: Gaussian with sigma={ic.params['sigma_x']}")

# FP configuration: pure diffusion (zero drift)
velocities = VelocitiesConfig(grid_1d, mu_x=0.0)  # Zero velocity (no drift)
diffusions = DiffusionConfig(grid_1d, axes=["x"], constants={"x": 0.5})

# Boundary conditions: periodic
from stochlib.boundary_conditions import BoundaryConditions

bcs = BoundaryConditions(grid_1d, bc_x="periodic")

logger.info("Configured FP: pure diffusion, periodic BC")

2026-01-19 23:45:52,814 [INFO    ] stochlib.notebook: Created 1D grid: 301 points
2026-01-19 23:45:52,814 [INFO    ] stochlib.notebook: Initial condition: Gaussian with sigma=1.0
2026-01-19 23:45:52,814 [INFO    ] stochlib.notebook: Configured FP: pure diffusion, periodic BC


In [9]:
# Create simulation engine and solve
from stochlib.fokker_planck import SimulationEngine

engine = SimulationEngine(grid_1d, velocities, diffusions, bcs)

# Time array for solving - with stable time stepping
# The CFL condition requires dt <= dx^2 / (2*D), so we use smaller dt
t_array = np.linspace(0, 0.3, 101)  # 0 to 0.3 time units with stable stepping

logger.info(f"Starting 1D solve: {len(t_array)-1} steps from t=0 to t={t_array[-1]:.3f}")
result = engine.run(f0=ic.f0, t_array=t_array, save_interval=10, confirm_run=False)
logger.info(f"Solve complete. Final time t={t_array[-1]:.3f}")

# Extract results
f_eval = [ic.f0] + result["snapshots"]
t_eval = result["times"]

2026-01-19 23:46:12,787 [INFO    ] stochlib.notebook: Starting 1D solve: 100 steps from t=0 to t=0.300
2026-01-19 23:46:12,788 [INFO    ] stochlib.fokker_planck.selector: Engine Decision: {'x': 'central_cn'}

               PRE-RUN SIMULATION REPORT

GRID CONFIGURATION:
  Active axes          : x
  Total grid points    : 301
  Grid deltas (Δ)      : {'x': np.float64(0.06666666666666643)}
  Volume element (dV)  : 6.667e-02

BOUNDARY CONDITIONS:
  x      : periodic

PHYSICS CONFIGURATION:
  Max velocity (|μ|)   : 0.000e+00
  Max diffusion (D)    : 5.000e-01

TIME STEPPING:
  dt (requested)       : 3.000e-03
  dt_max (CFL stable)  : 4.444e-03
  Total steps          : 100
  Time window          : [t=0.000e+00, t=3.000e-01]

NUMERICAL SCHEMES:
  x      : central_cn

DIAGNOSTICS:
  Report interval      : every 10 step(s)
  Mass tolerance       : 1.00e-03

RESOURCE ESTIMATES:
  Per-field memory     : 0.0000 GB
  Num fields           : 3 (f + velocities + diffusion)
  Est. total memory    : 0.

In [11]:
# Compute diagnostics
final_solution = result["final"]

mass = np.sum(final_solution) * grid_1d.volume_element
x = grid_1d.x_grid
mean = np.sum(x * final_solution) * grid_1d.dx
variance = np.sum((x - mean) ** 2 * final_solution) * grid_1d.dx

logger.info(f"Final diagnostics: mass={mass:.6f}, mean={mean:.4f}, variance={variance:.4f}")
print(f"Mass conservation: {mass:.8f} (should be ~1.0)")
print(f"Mean position: {mean:.6f}")
print(f"Variance: {variance:.6f} (initial was {ic.params['sigma_x']**2:.6f})")

2026-01-19 23:46:40,477 [INFO    ] stochlib.notebook: Final diagnostics: mass=1.000000, mean=-0.0000, variance=1.3000
Mass conservation: 1.00000000 (should be ~1.0)
Mean position: -0.000000
Variance: 1.300000 (initial was 1.000000)


In [ ]:
# Plot final distribution
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(grid_1d.x_grid, ic.f0, "b-", linewidth=2, label="Initial")
ax.plot(grid_1d.x_grid, result["final"], "r-", linewidth=2, label=f"Final (t={t_array[-1]:.2f})")
ax.set_xlabel("x", fontsize=12)
ax.set_ylabel("Probability density", fontsize=12)
ax.set_title("1D Diffusion: Initial vs Final Distribution", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

logger.info("Plotted 1D distribution")

## Part 2: 2D Advection-Diffusion Example

In [ ]:
# Create 2D grid
grid_2d = Grid(x_start=-5.0, x_end=5.0, num_points_x=101, y_start=-5.0, y_end=5.0, num_points_y=101)
logger.info(f"Created 2D grid: {grid_2d.num_points_x}x{grid_2d.num_points_y} points")

# Initial condition: 2D Gaussian
ic_2d = InitialCondition(grid_2d, func_type="gaussian", x0=0.0, sigma_x=0.8, y0=0.0, sigma_y=0.8)

# FP configuration: drift in x + diffusion in both directions
# Constant drift in x direction
velocities_2d = VelocitiesConfig(grid_2d, mu_x=0.2, mu_y=0.0)
# Isotropic diffusion
diffusions_2d = DiffusionConfig(grid_2d, axes=["x", "y"], constants={"x": 0.1, "y": 0.1})

# Boundary conditions: noflux (reflecting walls)
bcs_2d = BoundaryConditions(grid_2d, bc_x="noflux", bc_y="noflux")

logger.info("Configured 2D FP: advection + diffusion, noflux BC")

In [ ]:
# Create 2D simulation engine and solve
engine_2d = SimulationEngine(grid_2d, velocities_2d, diffusions_2d, bcs_2d)

# Time array for 2D
t_array_2d = np.linspace(0, 0.5, 51)  # 0 to 0.5, 50 steps

logger.info(f"Starting 2D solve: {len(t_array_2d)-1} steps")
result_2d = engine_2d.run(f0=ic_2d.f0, t_array=t_array_2d, save_interval=5)
logger.info(f"2D solve complete. Final time t={t_array_2d[-1]:.3f}")

In [ ]:
# Plot 2D distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Initial distribution
im0 = axes[0].contourf(grid_2d.X, grid_2d.Y, ic_2d.f0, levels=20, cmap="viridis")
axes[0].set_xlabel("x", fontsize=11)
axes[0].set_ylabel("y", fontsize=11)
axes[0].set_title("Initial Distribution", fontsize=12)
axes[0].set_aspect("equal")
plt.colorbar(im0, ax=axes[0])

# Final distribution
im1 = axes[1].contourf(grid_2d.X, grid_2d.Y, result_2d["final"], levels=20, cmap="viridis")
axes[1].set_xlabel("x", fontsize=11)
axes[1].set_ylabel("y", fontsize=11)
axes[1].set_title(f"Final Distribution (t={t_array_2d[-1]:.2f})", fontsize=12)
axes[1].set_aspect("equal")
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

logger.info("Plotted 2D distributions")

## Summary

This notebook demonstrated:
1. Setting up 1D and 2D grids with FP configurations
2. Solving FP PDEs with different velocity and diffusion profiles
3. Computing diagnostics (mass, entropy)
4. Visualizing probability distributions
5. Using logging to track progress